# Which New York departures will leave 15+ minutes late?
All 336,776 flights that were scheduled to leave JFK, LaGuardia and Newark in 2013
(US Bureau of Transportation Statistics on-time data, packaged as `nycflights13`).

Run from the project folder:  `python src/analysis.py`   (or open in Spyder / VS Code and run cell by cell)

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
FIG, RES = ROOT / "figures", ROOT / "results"
FIG.mkdir(exist_ok=True), RES.mkdir(exist_ok=True)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8982"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e4dd", "axes.axisbelow": True,
                     "font.size": 10.5, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})
funnel = []                                     # every cleaning step is logged here: (label, rows, note)


def log(label, df, note):
    funnel.append({"label": label, "rows": int(len(df)), "note": note})
    print(f"{label:<45s} {len(df):>8,d} rows   {note}")

## 1. Load
`pip install nycflights13` ships the five tables as CSV. Nothing is downloaded at run time.

In [2]:
import nycflights13 as nyc                      # noqa: E402

flights, weather, planes, airlines = nyc.flights.copy(), nyc.weather.copy(), nyc.planes.copy(), nyc.airlines.copy()
log("Raw flights", flights, "every scheduled departure from JFK, LGA, EWR in 2013")
print(flights.isna().sum()[lambda s: s > 0])

Raw flights                                    336,776 rows   every scheduled departure from JFK, LGA, EWR in 2013
dep_time     8255
dep_delay    8255
arr_time     8713
arr_delay    9430
tailnum      2512
air_time     9430
dtype: int64


## 2. Clean
### 2.1 Missing departure times are cancelled flights, not random gaps
`dep_time` and `dep_delay` are empty for 8,255 flights. They are empty because the flight never left.
`dropna()` would delete them silently and nobody would know cancellations existed. They are counted
first, then set aside, because a flight that never departs has no departure delay to predict.

In [3]:
flights["cancelled"] = flights["dep_time"].isna()
n_cancel = int(flights["cancelled"].sum())
cancel_by_month = flights.groupby("month")["cancelled"].mean().mul(100).round(2)
print(f"Cancelled: {n_cancel:,} of {len(flights):,} ({n_cancel / len(flights):.2%})")
df = flights.loc[~flights["cancelled"]].copy()
log("Departed flights (cancellations set aside)", df, f"{n_cancel:,} cancelled flights analysed separately, not deleted")

Cancelled: 8,255 of 336,776 (2.45%)
Departed flights (cancellations set aside)     328,521 rows   8,255 cancelled flights analysed separately, not deleted


### 2.2 Types: timestamps arrive as text
`time_hour` is a string such as `2013-01-01T10:00:00Z`. As text it can be matched but not shifted or
sorted in time, so it is converted once, in both tables, before any join.

In [4]:
for t in (df, weather):
    t["time_hour"] = pd.to_datetime(t["time_hour"], utc=True, errors="coerce")
assert df["time_hour"].notna().all() and weather["time_hour"].notna().all()      # coerce created no NaT
df["sched"] = pd.to_datetime(dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute))
df["dow"] = df["sched"].dt.dayofweek           # 0 = Monday
df["late15"] = (df["dep_delay"] >= 15).astype(int)   # US DOT definition of a delayed flight
print("Share of departed flights 15+ min late:", round(df["late15"].mean(), 4))

Share of departed flights 15+ min late: 0.2219


### 2.3 Weather: an impossible value and two very different missing columns
* One wind speed is 1,048 mph. The strongest surface gust ever recorded is about 253 mph, so this is an
  instrument or entry error. It becomes NaN (unknown), not 0 and not the mean.
* `wind_gust` is empty 80% of the time: the column is dropped. `pressure` is empty 10% of the time:
  it is kept and imputed inside the modelling pipeline (median learned from training rows only).

In [5]:
print(weather.isna().mean().round(3)[lambda s: s > 0])
bad_wind = weather["wind_speed"] > 100
print("Impossible wind speeds set to NaN:", int(bad_wind.sum()), weather.loc[bad_wind, "wind_speed"].round(0).tolist())
weather.loc[bad_wind, "wind_speed"] = np.nan
weather = weather.drop(columns=["wind_gust", "year", "month", "day", "hour"])
wx_cols = ["temp", "dewp", "humid", "wind_speed", "precip", "pressure", "visib"]

# The weather a dispatcher KNOWS one hour before departure is the reading from one hour earlier.
# Shifting the key by +1h means: the row stamped 09:00 is used for flights scheduled at 10:00.
wx_known = weather[["origin", "time_hour"] + wx_cols].copy()
wx_known["time_hour"] = wx_known["time_hour"] + pd.Timedelta(hours=1)
before = len(df)
df = df.merge(wx_known, on=["origin", "time_hour"], how="left", validate="many_to_one")
assert len(df) == before                        # a left join must never change the number of flights
print("Flights with no weather reading one hour earlier:", int(df["temp"].isna().sum()))

wind_dir     0.018
wind_gust    0.796
pressure     0.104
dtype: float64
Impossible wind speeds set to NaN: 1 [1048.0]
Flights with no weather reading one hour earlier: 1334


### 2.4 Aircraft age: 16% of tail numbers are not in the registry table
Kept as missing with a flag. Deleting them would remove most of two carriers' flights.

In [6]:
planes = planes.rename(columns={"year": "plane_year"})[["tailnum", "plane_year", "seats"]]
df = df.merge(planes, on="tailnum", how="left", validate="many_to_one")
df["plane_age"] = 2013 - df["plane_year"]
df.loc[(df["plane_age"] < 0) | (df["plane_age"] > 60), "plane_age"] = np.nan
miss_plane = df["plane_age"].isna()
print(f"Plane age unknown for {miss_plane.mean():.1%} of flights; by carrier (top 4):")
print(df.groupby("carrier")["plane_age"].apply(lambda s: s.isna().mean()).sort_values(ascending=False).head(4).round(3))
log("After joins (weather, aircraft)", df, "left joins: row count unchanged, gaps kept as NaN and flagged")

Plane age unknown for 16.4% of flights; by carrier (top 4):


carrier
MQ    0.962
AA    0.695
HA    0.094
FL    0.086
Name: plane_age, dtype: float64
After joins (weather, aircraft)                328,521 rows   left joins: row count unchanged, gaps kept as NaN and flagged


### 2.5 Features that describe the airport's state, using only what is already known
Delays spread through the day. Two features capture that without looking into the future:
* `sched_dep_hour_count`: how many departures the airport has scheduled in that hour (known months ahead).
* `earlier_delay`: average delay of flights from the same airport scheduled 3 to 6 hours earlier, with each
  delay capped at 120 minutes. One hour before our departure, any flight scheduled 3+ hours ago has either
  left (delay known) or is known to be at least 120 minutes late, so this number is fully observable.

In [7]:
df["sched_dep_hour_count"] = df.groupby(["origin", "time_hour"])["flight"].transform("size")
hourly = (df.assign(d=df["dep_delay"].clip(upper=120)).groupby(["origin", "time_hour"])["d"].agg(["sum", "count"]))
parts = []
for origin, g in hourly.groupby(level=0):
    g = g.droplevel(0)
    full = pd.date_range(g.index.min(), g.index.max(), freq="h")          # every hour of the year, even empty ones
    g = g.reindex(full, fill_value=0)                                     # reindex exposes the hours with no flights
    win = g.rolling(3).sum().shift(3)                                     # hours t-5, t-4, t-3
    out = (win["sum"] / win["count"].where(win["count"] > 0)).rename("earlier_delay").to_frame()
    out["origin"] = origin
    parts.append(out.rename_axis("time_hour").reset_index())
df = df.merge(pd.concat(parts), on=["origin", "time_hour"], how="left", validate="many_to_one")
print("earlier_delay missing (first departures of the day):", f"{df['earlier_delay'].isna().mean():.1%}")

earlier_delay missing (first departures of the day): 15.5%


## 3. Explore
Each chart answers one question. Read them with the six moves: frame, pattern, exceptions, compare, meaning, doubt.

In [8]:
by_hour = df.groupby("hour")["late15"].agg(["mean", "size"]).query("size >= 500")
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(by_hour.index, by_hour["mean"] * 100, color=BLUE, width=0.75)
ax.axhline(df["late15"].mean() * 100, color=GREY, ls="--", lw=1)
ax.text(by_hour.index.max() + 0.4, df["late15"].mean() * 100, " all-day\n average", va="center", color=GREY, fontsize=9)
ax.set(xlabel="Scheduled departure hour (local time)", ylabel="% of flights leaving 15+ min late",
       title="The later in the day, the more likely a late departure", xticks=by_hour.index)
fig.savefig(FIG / "fig1_delay_by_hour.png"); plt.close(fig)

In [9]:
by_month = df.groupby("month")["late15"].mean() * 100
fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
axes[0].plot(by_month.index, by_month.values, color=BLUE, marker="o", lw=2)
axes[0].set(ylabel="% late 15+ min", title="Summer storms and December drive delays; February drives cancellations", ylim=(0, None))
axes[1].bar(cancel_by_month.index, cancel_by_month.values, color=ORANGE, width=0.6)
axes[1].set(ylabel="% cancelled", xlabel="Month of 2013", xticks=range(1, 13))
fig.savefig(FIG / "fig2_month.png"); plt.close(fig)

In [10]:
by_carrier = (df.groupby("carrier")["late15"].agg(["mean", "size"]).join(airlines.set_index("carrier")).query("size >= 3000").sort_values("mean"))
by_carrier["name"] = by_carrier["name"].str.replace(r" (Air Lines|Airlines|Airways|Air) ?(Inc\.|Co\.|Corporation)?$", "", regex=True)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(by_carrier["name"], by_carrier["mean"] * 100, color=BLUE, height=0.7)
for y, (m, n) in enumerate(zip(by_carrier["mean"], by_carrier["size"])):
    ax.text(m * 100 + 0.4, y, f"{m:.0%}  (n={n:,})", va="center", fontsize=8.5, color="#52514e")
ax.set(xlabel="% of flights leaving 15+ min late", title="Delay rate by carrier (carriers with 3,000+ departures)", xlim=(0, by_carrier["mean"].max() * 125))
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig3_carrier.png"); plt.close(fig)

In [11]:
bins = [-np.inf, 5, 10, 20, 30, 45, np.inf]
labels = ["<5", "5-10", "10-20", "20-30", "30-45", "45+"]
by_state = df.dropna(subset=["earlier_delay"]).groupby(pd.cut(df["earlier_delay"], bins=bins, labels=labels), observed=True)["late15"].agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(by_state.index.astype(str), by_state["mean"] * 100, color=BLUE, width=0.7)
for x, (m, n) in enumerate(zip(by_state["mean"], by_state["size"])):
    ax.text(x, m * 100 + 1, f"{m:.0%}\nn={n:,}", ha="center", fontsize=8.5, color="#52514e")
ax.set(xlabel="Average delay (min) of flights scheduled 3 to 6 hours earlier at the same airport", ylabel="% late 15+ min",
       title="Delays propagate: a late airport stays late", ylim=(0, by_state["mean"].max() * 118))
fig.savefig(FIG / "fig4_propagation.png"); plt.close(fig)

rain = df.assign(rain=np.where(df["precip"] > 0, "rain or snow in the hour before", "dry"))[lambda d: d["temp"].notna()].groupby("rain")["late15"].agg(["mean", "size"])
print(rain)

                                     mean    size
rain                                             
dry                              0.208465  306503
rain or snow in the hour before  0.420663   20684


## 4. Split by time, not at random
The model will be used on future flights. So it is trained on January to September and tested on
October to December, which it has never seen. A random split is also run, only to show how much
more flattering it looks.

In [12]:
cat_cols = ["carrier", "origin", "dest"]
sched_num = ["month", "dow", "hour", "distance", "sched_dep_hour_count"]            # known when the schedule is published
ops_num = wx_cols + ["plane_age", "earlier_delay"]                                   # known about an hour before departure
df = df.sort_values("sched").reset_index(drop=True)
train, test = df[df["month"] <= 9], df[df["month"] >= 10]
y_train, y_test = train["late15"], test["late15"]
print(f"train {len(train):,} rows (Jan-Sep, {y_train.mean():.1%} late) | test {len(test):,} rows (Oct-Dec, {y_test.mean():.1%} late)")

train 245,723 rows (Jan-Sep, 23.1% late) | test 82,798 rows (Oct-Dec, 19.6% late)


## 5. Models
* **Baseline**: give every flight the training-period delay rate. Any model must beat this.
* **Logistic regression**: readable, needs scaling and one-hot encoding.
* **Gradient boosting** (`HistGradientBoostingClassifier`): handles curves, interactions and missing values natively.

Each is fitted twice: with **schedule-time** features only (what a planner knows months ahead), and with
**day-of-operations** features added (weather an hour before, aircraft age, how late the airport already is).
All imputing, scaling and encoding sit inside a Pipeline, so they are learned from the training months only.

In [13]:
def logit(num):
    pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
                             ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=500), cat_cols)])
    return make_pipeline(pre, LogisticRegression(max_iter=400, C=1.0))


def gbm(num):
    pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat_cols), ("num", "passthrough", num)])
    return make_pipeline(pre, HistGradientBoostingClassifier(categorical_features=list(range(len(cat_cols))), max_iter=300, learning_rate=0.08,
                                                             early_stopping=True, validation_fraction=0.15, random_state=42))


def score(name, p, y):
    return {"name": name, "auc": roc_auc_score(y, p), "ap": average_precision_score(y, p), "brier": brier_score_loss(y, p)}


results, fitted = [], {}
base_p = np.full(len(test), y_train.mean())
results.append({"name": "Baseline: always the average rate", "auc": 0.5, "ap": float(y_test.mean()), "brier": brier_score_loss(y_test, base_p)})
for label, num in [("schedule-time", sched_num), ("day-of-operations", sched_num + ops_num)]:
    for mname, make in [("Logistic regression", logit), ("Gradient boosting", gbm)]:
        m = make(num).fit(train[cat_cols + num], y_train)
        p = m.predict_proba(test[cat_cols + num])[:, 1]
        results.append(score(f"{mname}, {label}", p, y_test)); fitted[(mname, label)] = (m, num, p)
        print(results[-1])

# How much would a random split have flattered the best model?
num = sched_num + ops_num
Xa, Xb, ya, yb = train_test_split(df[cat_cols + num], df["late15"], test_size=0.25, random_state=42, stratify=df["late15"])
auc_random = roc_auc_score(yb, gbm(num).fit(Xa, ya).predict_proba(Xb)[:, 1])
print(f"Same model, random split AUC = {auc_random:.3f}  vs  time split AUC = {results[-1]['auc']:.3f}")

{'name': 'Logistic regression, schedule-time', 'auc': 0.6685254031256518, 'ap': 0.3039216261858505, 'brier': 0.1523197160284946}


{'name': 'Gradient boosting, schedule-time', 'auc': 0.6723833080887551, 'ap': 0.3149569509239775, 'brier': 0.15177482845100787}


{'name': 'Logistic regression, day-of-operations', 'auc': 0.719136461715838, 'ap': 0.44018258156131584, 'brier': 0.1403207454315567}


{'name': 'Gradient boosting, day-of-operations', 'auc': 0.7364786812500024, 'ap': 0.4654517388277973, 'brier': 0.13443071930019276}


Same model, random split AUC = 0.774  vs  time split AUC = 0.736


In [14]:
res = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
cols = [GREY, BLUE, BLUE, ORANGE, ORANGE]
ax.barh(res["name"][::-1], res["auc"][::-1], color=cols[::-1], height=0.65)
for y, v in enumerate(res["auc"][::-1]):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9)
ax.set(xlim=(0.45, 0.85), xlabel="ROC AUC on October to December (unseen)", title="Knowing the state of the day is worth more than a cleverer model")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig5_models.png"); plt.close(fig)

## 6. Is the probability trustworthy, and what drives it?

In [15]:
best_m, best_num, best_p = fitted[("Gradient boosting", "day-of-operations")]
cal = pd.DataFrame({"p": best_p, "y": y_test.values})
cal["bin"] = pd.qcut(cal["p"], 10, duplicates="drop")
calib = cal.groupby("bin", observed=True).agg(pred=("p", "mean"), actual=("y", "mean"), n=("y", "size"))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
ax.plot(calib["pred"], calib["actual"], marker="o", color=ORANGE, lw=2)
ax.set(xlabel="Predicted probability of a 15+ min delay", ylabel="Share actually delayed", title="Calibration (ten equal-sized groups)",
       xlim=(0, max(0.85, calib["pred"].max() + .05)), ylim=(0, max(0.85, calib["actual"].max() + .05)))
fig.savefig(FIG / "fig6_calibration.png"); plt.close(fig)
print(calib.round(3))

                   pred  actual     n
bin                                  
(0.0102, 0.0491]  0.037   0.051  8280
(0.0491, 0.0679]  0.059   0.079  8281
(0.0679, 0.0869]  0.077   0.092  8279
(0.0869, 0.107]   0.097   0.123  8279
(0.107, 0.128]    0.118   0.139  8280
(0.128, 0.155]    0.141   0.163  8280
(0.155, 0.19]     0.171   0.184  8279
(0.19, 0.247]     0.216   0.234  8280
(0.247, 0.373]    0.299   0.325  8280
(0.373, 0.963]    0.553   0.572  8280


In [16]:
samp = test.sample(20000, random_state=1)
pi = permutation_importance(best_m, samp[cat_cols + best_num], samp["late15"], scoring="roc_auc", n_repeats=5, random_state=1, n_jobs=2)
imp = pd.Series(pi.importances_mean, index=cat_cols + best_num).sort_values(ascending=False)
print(imp.round(4))
nice = {"earlier_delay": "How late the airport already is", "hour": "Scheduled hour", "carrier": "Carrier", "dest": "Destination", "month": "Month",
        "sched_dep_hour_count": "Departures scheduled that hour", "humid": "Humidity", "visib": "Visibility", "precip": "Rain/snow last hour", "temp": "Temperature",
        "pressure": "Air pressure", "dewp": "Dew point", "wind_speed": "Wind speed", "dow": "Day of week", "origin": "Airport", "distance": "Distance", "plane_age": "Aircraft age"}
fig, ax = plt.subplots(figsize=(7.5, 4.0))
top = imp.head(10)[::-1]
ax.barh([nice[k] for k in top.index], top.values, color=BLUE, height=0.7)
ax.set(xlabel="Drop in AUC when the feature is shuffled (bigger = model relies on it more)", title="What the model relies on")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig7_importance.png"); plt.close(fig)

earlier_delay           0.0811
hour                    0.0586
carrier                 0.0319
dest                    0.0090
precip                  0.0042
humid                   0.0037
dow                     0.0033
wind_speed              0.0024
sched_dep_hour_count    0.0015
origin                  0.0011
visib                   0.0010
plane_age               0.0010
pressure                0.0010
temp                    0.0008
distance                0.0003
month                   0.0000
dewp                   -0.0011
dtype: float64


## 7. From a probability to a decision
Operations cannot protect every flight. If the team can act on the riskiest 10% of departures,
how many of the delays do they catch, compared with picking flights at random?

In [17]:
order = np.argsort(-best_p)
yt = y_test.values[order]
table = []
for share in (0.05, 0.10, 0.20, 0.30):
    k = int(len(yt) * share)
    table.append({"flag_top": share, "flights_flagged": k, "precision": float(yt[:k].mean()), "recall": float(yt[:k].sum() / yt.sum()), "lift": float(yt[:k].mean() / yt.mean())})
print(pd.DataFrame(table).round(3))

   flag_top  flights_flagged  precision  recall   lift
0      0.05             4139      0.694   0.177  3.536
1      0.10             8279      0.572   0.292  2.917
2      0.20            16559      0.449   0.457  2.287
3      0.30            24839      0.377   0.577  1.922


In [18]:
edges = np.linspace(0, 1, 51)
pos, _ = np.histogram(best_p[y_test.values == 1], bins=edges)
neg, _ = np.histogram(best_p[y_test.values == 0], bins=edges)
out = {
    "n_raw": int(len(flights)), "n_cancelled": n_cancel, "n_model": int(len(df)), "late_rate": float(df["late15"].mean()),
    "mean_delay": float(df["dep_delay"].mean()), "median_delay": float(df["dep_delay"].median()),
    "wind_outlier": 1048, "plane_age_missing": float(miss_plane.mean()), "earlier_delay_missing": float(df["earlier_delay"].isna().mean()),
    "pressure_missing": float(df["pressure"].isna().mean()), "weather_missing_rows": int(df["temp"].isna().sum()),
    "train": {"n": int(len(train)), "rate": float(y_train.mean())}, "test": {"n": int(len(test)), "rate": float(y_test.mean())},
    "models": results, "auc_random_split": float(auc_random), "decision": table, "funnel": funnel,
    "by_hour": {int(k): float(v) for k, v in by_hour["mean"].items()}, "by_month": {int(k): float(v) for k, v in by_month.items()},
    "cancel_by_month": {int(k): float(v) for k, v in cancel_by_month.items()},
    "by_carrier": [{"name": r["name"], "rate": float(r["mean"]), "n": int(r["size"])} for _, r in by_carrier.iterrows()],
    "by_state": [{"bin": str(k), "rate": float(r["mean"]), "n": int(r["size"])} for k, r in by_state.iterrows()],
    "rain": {k: {"rate": float(r["mean"]), "n": int(r["size"])} for k, r in rain.iterrows()},
    "calibration": [{"pred": float(r.pred), "actual": float(r.actual), "n": int(r.n)} for r in calib.itertuples()],
    "importance": [{"name": nice[k], "value": float(v)} for k, v in imp.items()],
    "bins": {"pos": pos.tolist(), "neg": neg.tolist()},
}
(RES / "metrics.json").write_text(json.dumps(out, indent=1))
print("Saved figures/ and results/metrics.json")

Saved figures/ and results/metrics.json
